# FLIC: Data Analysis Part 3 — Analysis of Choice Experiments
**Scott Pletcher and the Pletcher Lab**

---

## Overview

This notebook introduces functions designed to visualise and quantify preference information when individual flies are placed in chambers containing two food choices.  It is the Python port of the R document *ChoiceChamberAnalysis.Rmd* (Part 3 of the FLIC documentation series).

The **preference index (PI)** is the central metric in a choice experiment.  It is defined as:

$$PI = \frac{WellA - WellB}{WellA + WellB}$$

where *WellA* and *WellB* are the total lick (or event) counts in each logical well over the analysis window.  A PI of +1 indicates exclusive feeding from WellA; −1 indicates exclusive feeding from WellB; 0 indicates equal preference.  **PI is always calculated from raw, untransformed lick counts** — the 0.25-power lick transformation used elsewhere for plotting does not affect PI.

**WellA and WellB are logical labels**, not physical positions.  The mapping between the physical left/right well and WellA/WellB is controlled by the `pi_direction` field in `flic_config.yaml` (see below).  This replaces R's `PI.Multiplier` parameter and achieves the same counterbalancing: setting `pi_direction: left` on some DFMs and `pi_direction: right` on others ensures that a consistent food (e.g. Sucrose) is always called WellA across all monitors, regardless of which physical side the food occupies.

PI-related methods are provided by `TwoWellExperiment` and its subclasses (`HedonicFeedingExperiment`, `ProgressiveRatioExperiment`).  `Experiment.load()` automatically returns the correct subclass based on the `experiment_type` field in `flic_config.yaml`; the test data (a standard two-well choice experiment) returns a `TwoWellExperiment`.

This notebook covers:

- The WellA / WellB / pi_direction convention
- Cumulative PI per DFM — lick-based and event-based
- Accessing the underlying cumulative PI data frames
- Treatment-level PI summary plots
- Binned PI timecourses
- Experiment-wide cumulative PI plots (all DFMs)
- Statistical analysis of PI using `statsmodels`

Lick counts, event counts, event durations, inter-event intervals, and tasting metrics are discussed in the companion **Grouped Analysis** notebook (Part 2).

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
from pathlib import Path

# Locate the pyflic repository root (the directory that contains pyproject.toml).
def _find_repo_root(marker: str = "pyproject.toml") -> Path:
    for p in [Path().resolve(), *Path().resolve().parents]:
        if (p / marker).exists():
            return p
    raise RuntimeError(f"Could not find repo root ({marker} not found in any parent).")

repo_root = _find_repo_root()
if str(repo_root.parent) not in sys.path:
    sys.path.insert(0, str(repo_root.parent))

# The three classes you will use most often.
from pyflic import Experiment, DFM, Parameters

project_dir = repo_root / "python_test_data"
exp = Experiment.load(project_dir)
print("DFMs loaded:", sorted(exp.dfms.keys()))
from IPython.display import display


## WellA, WellB, and pi_direction

Before analysing PI it is important to understand three terms that appear throughout this notebook and in the data output.

**Well Number** refers to the physical position of a food well on the DFM.  All DFM versions number wells 1–12 from left to right, top to bottom.  This number never changes and is independent of lid type or experiment design.

**Chamber** refers to the physical space occupied by a fly (or small group of flies).  With the standard six-chamber choice lid each chamber contains two food wells — one on the left and one on the right.  Chambers are numbered 1–6 from top to bottom.

**WellA** and **WellB** are *logical* labels.  In a choice experiment a PI of +1 means the fly fed exclusively from WellA, and a PI of −1 means it fed exclusively from WellB.  Which physical well is called WellA is determined by the `pi_direction` parameter:

| `pi_direction` | WellA (physical) | WellB (physical) |
|---|---|---|
| `"left"` | Left well of chamber | Right well of chamber |
| `"right"` | Right well of chamber | Left well of chamber |

This replaces R's `PI.Multiplier` parameter.  In R, the experimenter constructed two separate parameter objects — one with `PI.Multiplier = +1` and one with `PI.Multiplier = -1` — and supplied a per-DFM list to `Feeding.Summary.Monitors()`.  In pyflic, `pi_direction` is set once per DFM in `flic_config.yaml` and applied automatically.  It is impossible to misalign the list.

**In the test data** the food positions are counterbalanced: Sucrose is in the left well on DFMs 1, 2, 5, 6 and in the right well on DFMs 3, 4, 7, 8.  To ensure that WellA always maps to Sucrose and WellB always maps to Yeast, `pi_direction` is set to `"left"` on DFMs 1, 2, 5, 6 and `"right"` on DFMs 3, 4, 7, 8.  As a result, a positive PI in every DFM indicates preference for Sucrose regardless of the physical side.

`exp.design.design_table()` shows the complete assignment of treatments to (DFM, Chamber) pairs.

In [ ]:
exp.design.design_table().head(12)

In [ ]:
# Confirm pi_direction for each DFM.
for dfm_id, dfm in sorted(exp.dfms.items()):
    print(f"DFM {dfm_id}: pi_direction={dfm.params.pi_direction!r}")

## Cumulative PI — Per DFM

To visualise how preference evolves over the course of an experiment, use `dfm.plot_cumulative_pi()`.  This function plots the running cumulative PI for each chamber in a single DFM.

Cumulative PI is initialised to 0 at the start of the analysis window (or at the start of the experiment) and updates at each lick event.  Each point on the plot corresponds to a time at which at least one lick was recorded; flat line segments connecting adjacent points indicate periods of inactivity.  The colour of each point encodes the total number of licks that have contributed to the PI at that moment, making it easy to distinguish noisy low-interaction trajectories from well-sampled ones.

By default one subplot is drawn per chamber (`single_plot=False`).  Setting `single_plot=True` overlays all chambers on a single axes, with one line per chamber distinguished by colour.

The function returns a matplotlib `Figure`; the underlying data are available via `dfm.cumulative_pi_data()`.

| R function | Python equivalent |
|---|---|
| `CumulativePI.DFM(dfm)` | `dfm.plot_cumulative_pi()` |
| `CumulativePI.DFM(dfm, SinglePlot=TRUE)` | `dfm.plot_cumulative_pi(single_plot=True)` |

In [ ]:
dfm1 = exp.get_dfm(1)

# Default: one subplot per chamber.
fig = dfm1.plot_cumulative_pi()
display(fig)

Chambers with few lick interactions have noisier PI trajectories — their lines jump between extreme values (+1 and −1) because each individual lick has an outsized influence on the cumulative PI.  Chambers with sustained feeding activity produce smoother trajectories that converge toward a stable preference.  Colour intensity encodes cumulative lick count, so faint early portions of a trace reflect low confidence.

To overlay all chambers on a single axes — useful when you want to compare trajectories side by side — use `single_plot=True`.  In this mode each chamber is drawn with a distinct colour rather than encoding lick count by colour.

In [ ]:
# Overlay all chambers on one axes; colour identifies chamber.
fig = dfm1.plot_cumulative_pi(single_plot=True)
display(fig)

### Focusing on a Time Range

The `range_minutes` argument restricts the analysis to a specific window.  Importantly, the cumulative PI is **reset to 0 at the start of the specified window**, so all feeding that occurred before the window is ignored.  This lets you examine whether a preference that was established early in the experiment is maintained, strengthened, or reversed later on.

| R function | Python equivalent |
|---|---|
| `CumulativePI.DFM(dfm, range=c(200, 300))` | `dfm.plot_cumulative_pi(range_minutes=(200, 300))` |

In [ ]:
# Restrict to the first 180 minutes; PI resets to 0 at t=0.
fig = dfm1.plot_cumulative_pi(range_minutes=(0, 180))
display(fig)

### Accessing the Underlying Data

`dfm.cumulative_pi_data()` returns the same data used to produce the plot as a tidy DataFrame.  Unlike the raw data stored on the `DFM` object (which has one row per sample), this DataFrame has one row per lick event — i.e. only time points at which at least one feeding lick was recorded are included.

| R function | Python equivalent |
|---|---|
| `CumulativePI.DFM(dfm, ShowPlots=FALSE)` | `dfm.cumulative_pi_data()` |

In [ ]:
pi_data = dfm1.cumulative_pi_data()
pi_data.head(10)

| Column | Definition |
|---|---|
| `Minutes` | Elapsed time (minutes) at which a lick was recorded |
| `PI` | Cumulative PI at that time point |
| `Licks` | Total lick count (WellA + WellB) used to compute this PI value |
| `Chamber` | Chamber number (1–6) |

## Cumulative Event PI — Per DFM

`dfm.plot_cumulative_event_pi()` works identically to `plot_cumulative_pi()` but counts feeding *events* (bouts) rather than individual licks.  The two metrics will often track closely, but can diverge when one well produces many short events while the other produces fewer but longer ones — in that case lick-based PI and event-based PI may tell different stories.

Two additional arguments are available:

- **`events_limit`** — restrict the analysis to the first N events per chamber.  Useful when you want to compare behaviour after an equal number of feeding experiences, regardless of when in the experiment they occurred.
- **`by_bout`** — plot event number on the x-axis instead of elapsed time.  This highlights whether preference is dependent on feeding *experience* (number of bouts) rather than clock time.

| R function | Python equivalent |
|---|---|
| `CumulativeEventPI.DFM(dfm)` | `dfm.plot_cumulative_event_pi()` |
| `CumulativeEventPI.DFM(dfm, events.limit=50)` | `dfm.plot_cumulative_event_pi(events_limit=50)` |
| `CumulativeEventPI.DFM(dfm, by.bout=TRUE)` | `dfm.plot_cumulative_event_pi(by_bout=True)` |

In [ ]:
fig = dfm1.plot_cumulative_event_pi()
display(fig)

In [ ]:
# Restrict to the first 50 feeding events per chamber.
fig = dfm1.plot_cumulative_event_pi(events_limit=50)
display(fig)

In [ ]:
# Plot by bout number rather than elapsed time.
fig = dfm1.plot_cumulative_event_pi(events_limit=50, by_bout=True)
display(fig)

When `by_bout=True`, not all chambers will necessarily reach the `events_limit` value: chambers where the fly fed fewer than `events_limit` times contribute only up to their actual event count.  Interpret the right-hand end of each trajectory accordingly — a chamber whose line ends early contributed fewer total events.

The underlying data is available via `dfm.cumulative_event_pi_data()`.

In [ ]:
event_pi_data = dfm1.cumulative_event_pi_data()
event_pi_data.head(10)

| Column | Definition |
|---|---|
| `Minutes` | Elapsed time (minutes) at which the event was recorded |
| `PI` | Cumulative event PI at that point |
| `Chamber` | Chamber number (1–6) |
| `EventNum` | Cumulative event count used to compute this PI value |

## Treatment-Level PI Summary

`exp.feeding_summary()` returns a DataFrame with one row per (DFM, Chamber) that includes `PI` and `EventPI` columns alongside all lick, event, and duration metrics.  Because PI is always computed from raw (untransformed) lick counts, the `transform_licks` argument has no effect on the PI or EventPI columns — only on `LicksA`, `LicksB`, and related count columns.

Use `exp.plot_jitter_summary()` to visualise PI by treatment: each point represents one chamber, and the cross with error bars shows the mean ± SEM across chambers within each treatment.

| R function | Python equivalent |
|---|---|
| `DataPlot(f.summary2, Type="PI")` | `exp.plot_jitter_summary(fs, x_col="Treatment", y_col="PI")` |
| `DataPlot(f.summary2, Type="EventPI")` | `exp.plot_jitter_summary(fs, x_col="Treatment", y_col="EventPI")` |

In [ ]:
fs = exp.feeding_summary()
print("PI columns:", [c for c in fs.columns if "PI" in c])
fs[["Treatment", "DFM", "Chamber", "PI", "EventPI"]].head(12)

In [ ]:
# Lick-based PI by treatment.
p = exp.plot_jitter_summary(
    fs,
    x_col="Treatment",
    y_col="PI",
    title="Preference Index by Treatment",
    y_label="PI (Licks)",
)
p

In [ ]:
# Event-based PI by treatment.
p = exp.plot_jitter_summary(
    fs,
    x_col="Treatment",
    y_col="EventPI",
    title="Event Preference Index by Treatment",
    y_label="PI (Events)",
)
p

`exp.plot_feeding_summary()` also renders PI and EventPI panels — it produces a grid of box-and-jitter plots covering every metric from the feeding summary in a single figure, making it a convenient overview.  Refer to the **Grouped Analysis** notebook for a full discussion of `plot_feeding_summary()`.

## Binned PI Over Time

The cumulative PI (whether plotted over time or divided into periods) can mask time-dependent changes in preference because feeding from an early, dominant period dominates the cumulative total.  The most powerful way to detect preference shifts across the experiment is to compute the PI independently within non-overlapping time bins.

`exp.plot_binned_metric_by_treatment(metric="PI", binsize_min=30)` plots the mean ± SEM of the per-bin PI for each treatment across time.  Crucially, each bin's PI is computed from licks recorded only within that bin — cumulative history does not carry over between bins.  A treatment group whose PI is positive early and negative later changed its preference during the experiment.

| R function | Python equivalent |
|---|---|
| `BinnedDataPlot(f.binsummary2, Type="PI")` | `exp.plot_binned_metric_by_treatment(metric="PI", binsize_min=30)` |
| `BinnedDataPlot(f.binsummary2, Type="EventPI")` | `exp.plot_binned_metric_by_treatment(metric="EventPI", binsize_min=30)` |

In [ ]:
# Lick-based PI in 30-minute bins.
fig = exp.plot_binned_metric_by_treatment(metric="PI", binsize_min=30)
display(fig)

In [ ]:
# Event-based PI in 30-minute bins.
fig = exp.plot_binned_metric_by_treatment(metric="EventPI", binsize_min=30)
display(fig)

A positive PI in a given bin means that, within that 30-minute window, flies in that treatment group fed more from WellA (Sucrose) than from WellB (Yeast).  A PI near 0 means roughly equal feeding from both wells.  Inspect whether the PI is stable across bins (consistent preference), trends upward (preference strengthening over time), or trends downward and crosses zero (preference reversal).  The shaded region shows ± SEM across chambers within each treatment.

## Experiment-Wide Cumulative PI

Plotting cumulative PI across all DFMs gives a global view of individual-chamber behaviour throughout the entire experiment.  Using `single_plot=True` overlays all six chambers from each DFM on a single axes, making inter-chamber variability within a monitor immediately apparent.

Looping over all DFM IDs in `exp.dfms` replicates R's `CumulativePIPlots()` and `CumulativeEventPIPlots()` convenience functions.

| R function | Python equivalent |
|---|---|
| `CumulativePIPlots(monitors, params, expDesign)` | Loop over `exp.get_dfm(id).plot_cumulative_pi(single_plot=True)` |
| `CumulativeEventPIPlots(monitors, params, expDesign)` | Loop over `exp.get_dfm(id).plot_cumulative_event_pi(single_plot=True)` |
| `CumulativeEventPIPlots(..., events.limit=50, by.bout=TRUE)` | `dfm.plot_cumulative_event_pi(events_limit=50, by_bout=True)` |

In [ ]:
# Cumulative lick PI for every DFM (replicates R's CumulativePIPlots).
for dfm_id in sorted(exp.dfms.keys()):
    fig = exp.get_dfm(dfm_id).plot_cumulative_pi(single_plot=True)
    display(fig)

In [ ]:
# Cumulative event PI for every DFM (replicates R's CumulativeEventPIPlots).
for dfm_id in sorted(exp.dfms.keys()):
    fig = exp.get_dfm(dfm_id).plot_cumulative_event_pi(single_plot=True)
    display(fig)

## Statistical Analysis of PI

pyflic uses **`statsmodels`** for statistical tests, which mirrors R's `aov()` / `lm()` formula syntax closely.  For PI analysis:

- Use `PI` or `EventPI` directly from `exp.feeding_summary()` as the response variable — they are always computed from raw (untransformed) lick counts, so there is no need to pass `transform_licks=False`.
- For the binned ANOVA, call `exp.binned_feeding_summary()` with `save=False` to obtain the per-bin PI without writing files.
- Because `design_factors` are defined in `flic_config.yaml`, individual factor columns (`paired`, `genotype`) appear in the summary table automatically — no string manipulation required.

| R expression | Python equivalent |
|---|---|
| `summary(aov(PI ~ Treatment, data=f.summary2$Results))` | `anova_lm(smf.ols("PI ~ C(Treatment)", data=fs).fit())` |

Note: the binned ANOVA loop below does not correct for multiple comparisons.  Treat the per-interval p-values as exploratory and apply an appropriate correction (e.g. Bonferroni or Benjamini–Hochberg) when reporting results.

In [ ]:
import statsmodels.formula.api as smf
from statsmodels.stats.anova import anova_lm

# One-way ANOVA: PI ~ Treatment.
model = smf.ols("PI ~ C(Treatment)", data=fs).fit()
print(anova_lm(model))

In [ ]:
# Two-way ANOVA using individual design factor columns.
# typ=2 uses Type II sums of squares (recommended when sample sizes may be unequal).
model_2way = smf.ols("PI ~ C(paired) * C(genotype)", data=fs).fit()
print(anova_lm(model_2way, typ=2))

In [ ]:
# Binned ANOVA — test for treatment effect on PI in each 30-minute interval.
bfs = exp.binned_feeding_summary(binsize_min=30, save=False)

for interval in sorted(bfs["Interval"].unique())[:6]:
    subset = bfs[bfs["Interval"] == interval]
    model = smf.ols("PI ~ C(Treatment)", data=subset).fit()
    result = anova_lm(model)
    p = result["PR(>F)"].iloc[0]
    print(f"Interval {interval}: PI F p-value = {p:.4f}")

---

## Function Reference

Listed below are the principal functions introduced in this notebook with their signatures.  You should understand what each function does and what each argument represents before moving on.

### `DFM` — per-DFM PI functions

```python
dfm.cumulative_pi_data(
    *, range_minutes=(0, 0),
)                                            # → pd.DataFrame (Minutes, PI, Licks, Chamber)

dfm.plot_cumulative_pi(
    *, range_minutes=(0, 0),
    single_plot=False,
)                                            # → matplotlib Figure

dfm.cumulative_event_pi_data(
    *, events_limit=None,
    range_minutes=(0, 0),
)                                            # → pd.DataFrame (Minutes, PI, Chamber, EventNum)

dfm.plot_cumulative_event_pi(
    *, events_limit=None,
    range_minutes=(0, 0),
    single_plot=False,
    by_bout=False,
)                                            # → matplotlib Figure
```

### `Experiment` — treatment-level PI

```python
exp.feeding_summary(
    *, range_minutes=(0, 0),
    transform_licks=True,
)                                            # → pd.DataFrame  (includes PI and EventPI columns)

exp.plot_jitter_summary(
    df,
    *, x_col, y_col,
    facet_col="Treatment",
    title="", x_label=None, y_label=None,
    ylim=None, x_order=None, x_labels=None,
    colors=None, annotation=None,
    jitter_width=0.25, point_size=3.0,
    base_font_size=20.0,
)                                            # → plotnine.ggplot

exp.plot_binned_metric_by_treatment(
    *, metric="Licks",
    two_well_mode="total",
    binsize_min=30.0,
    range_minutes=(0, 0),
    transform_licks=True,
    show_sem=True,
    show_individual_chambers=False,
    figsize=(10, 4),
)                                            # → matplotlib Figure

exp.binned_feeding_summary(
    *, bins=None, binsize_min=None,
    range_minutes=(0, 0),
    transform_licks=True,
    path=None, save=True,
)                                            # → pd.DataFrame
```